In [0]:
-- ============================================================
-- GOLD DIMENSION: dim_category
-- PURPOSE:
--   Store product category values derived from category page URLs.
--
-- PRIMARY KEY:
--   category_id
--
-- RELATIONSHIP:
--   fact_marketing_performance.category_id
--     -> dim_category.category_id
--
-- SOURCE:
--   shoply_analytics.silver.page_views
--
-- CATEGORY RULE:
--   Category pages follow the pattern:
--     /category/<category_name>
--
-- NOTE:
--   An explicit 'unknown' member is added for sessions that
--   never visited a category page.
-- ============================================================

USE CATALOG shoply_analytics;
USE SCHEMA gold;

CREATE OR REPLACE VIEW dim_category AS
WITH extracted_categories AS (
    SELECT DISTINCT
        LOWER(TRIM(REGEXP_EXTRACT(page_url, '^/category/([^/?#]+)', 1))) AS category_name
    FROM shoply_analytics.silver.page_views
    WHERE page_url LIKE '/category/%'
),

all_categories AS (
    SELECT 'unknown' AS category_name
    UNION
    SELECT category_name
    FROM extracted_categories
    WHERE category_name IS NOT NULL
      AND category_name <> ''
)

SELECT
    -- Surrogate primary key
    ROW_NUMBER() OVER (
        ORDER BY category_name
    ) AS category_id,

    -- Category attribute
    category_name

FROM all_categories;